In [38]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# ==========================================
# 1. HARDWARE-SAFE DATA LOADING (64x64 Grayscale)
# ==========================================
datapath = r'/media/chethana/Paradox/FPGA/ML/Model/new_dataset' # Change this if running on Colab
img_size = (64, 64)
batch_size = 16

print("Initializing 64x64 Grayscale Regularized Pipeline...")

datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    validation_split=0.2
)

datagen_val = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# color_mode='grayscale' handles the 1-channel conversion automatically
train_ds = datagen_train.flow_from_directory(
    datapath, target_size=img_size, batch_size=batch_size,
    class_mode='sparse', subset='training', seed=123,
    color_mode='grayscale' 
)

val_ds = datagen_val.flow_from_directory(
    datapath, target_size=img_size, batch_size=batch_size,
    class_mode='sparse', subset='validation', seed=123, shuffle=False,
    color_mode='grayscale' 
)

class_names = list(train_ds.class_indices.keys())

Initializing 64x64 Grayscale Regularized Pipeline...


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\FPGA\\ML\\Model\\new_dataset'

In [ ]:
# ==========================================
# 2. THE FPGA BRAM ARCHITECTURE
# ==========================================
reg = regularizers.l2(0.001)

# Specific names applied to layers so they match the C++ variables perfectly
model = models.Sequential([
    layers.InputLayer(input_shape=(64, 64, 1)),

    layers.Conv2D(16, (3, 3), padding='same', activation='relu', kernel_regularizer=reg, name='conv1'),
    layers.BatchNormalization(name='bn1'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(32, (3, 3), padding='same', activation='relu', kernel_regularizer=reg, name='conv2'),
    layers.BatchNormalization(name='bn2'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu', kernel_regularizer=reg, name='conv3'),
    layers.BatchNormalization(name='bn3'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    
    layers.Dense(3, activation='softmax', name='dense1')
])

model.summary()

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
]

In [ ]:
# ==========================================
# 3. TRAINING & EVALUATION
# ==========================================
print("\n--- Starting Regularized Training ---")
history = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=callbacks)

print("\nGenerating final reports...")
val_ds.reset()
Y_pred = model.predict(val_ds)
y_pred_classes = np.argmax(Y_pred, axis=1)
y_true = val_ds.classes

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('64x64 Hardware-Optimized Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

Found 3340 images belonging to 3 classes.
Found 833 images belonging to 3 classes.


In [ ]:
# ==========================================
# 4. FUSION & C++ EXPORT
# ==========================================
print("\nExporting Fused Weights for FPGA...")
with open("cnn_weights.h", "w") as f:
    f.write("// Auto-Generated Hardware Weights (Batch Norm Fused)\n")
    f.write("#ifndef CNN_WEIGHTS_H\n#define CNN_WEIGHTS_H\n\n")

    for i, layer in enumerate(model.layers):
        if 'conv' in layer.name:
            print(f"Fusing {layer.name}...")
            conv_weights, conv_biases = layer.get_weights()[0], (layer.get_weights()[1] if len(layer.get_weights()) > 1 else np.zeros(layer.get_weights()[0].shape[-1]))
            
            next_layer = model.layers[i+1]
            if 'bn' in next_layer.name:
                print(f"  -> Found attached BN: {next_layer.name}")
                gamma, beta, moving_mean, moving_variance = next_layer.get_weights()
                epsilon = 1e-3 
                scale = gamma / np.sqrt(moving_variance + epsilon)
                fused_weights = conv_weights * scale
                fused_biases = (conv_biases - moving_mean) * scale + beta
            else:
                fused_weights, fused_biases = conv_weights, conv_biases

            flat_w, flat_b = fused_weights.flatten(), fused_biases.flatten()
            f.write(f"const float {layer.name}_w[{len(flat_w)}] = {{{', '.join(map(str, flat_w))}}};\n\n")
            f.write(f"const float {layer.name}_b[{len(flat_b)}] = {{{', '.join(map(str, flat_b))}}};\n\n")

        elif 'dense' in layer.name:
            print(f"Exporting {layer.name}...")
            dense_w, dense_b = layer.get_weights()
            flat_w, flat_b = dense_w.flatten(), dense_b.flatten()
            f.write(f"const float {layer.name}_w[{len(flat_w)}] = {{{', '.join(map(str, flat_w))}}};\n\n")
            f.write(f"const float {layer.name}_b[{len(flat_b)}] = {{{', '.join(map(str, flat_b))}}};\n\n")

    f.write("#endif\n")
print("SUCCESS: cnn_weights.h generated!")

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_out (Dense)               │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,939 (93.51 KB)

 Trainable params: 23,715 (92.64 KB)

 Non-trainable params: 224 (896.00 B)